# Làm quen với benchmark RCA

Notebook đọc kết quả đã chạy. Theo thứ tự: case → feature → split → trial → ranking → metric. Synthetic chỉ kiểm tra pipeline, chưa phản ánh mạng thật.

In [ ]:
from pathlib import Path
import json, csv
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
run = root / 'runs/demo_final/run'
prepared = root / 'runs/demo_final/prepared'
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8-sig'))
def read_jsonl(path):
    return [json.loads(s) for s in path.read_text(encoding='utf-8-sig').splitlines() if s]


## 1 Một case gồm những gì
`candidates` là tập cần xếp hạng. `root_entities` chỉ dùng làm nhãn; cutoff chốt thời điểm được nhìn thấy dữ liệu.

In [ ]:
cases = read_jsonl(prepared / 'cases.jsonl')
print(cases[-1])
print({s: sum(c['split']==s for c in cases) for s in ['train','validation','test']})


## 2 Feature và evidence
Mỗi row là một candidate trong một case; feature không chứa root label, fault type hoặc kết quả recovery.

In [ ]:
features = read_jsonl(prepared / 'features.jsonl')
print(features[-1])


## 3 Training và hyperparameter
So sánh validation trước; không dùng test để chọn cấu hình.

In [ ]:
with (run / 'trials.csv').open(encoding='utf-8-sig', newline='') as f:
    trials = list(csv.DictReader(f))
print('Trials:', len(trials))
print(trials[:2])


## 4 Đọc ranking và kết quả
Score cao chỉ là nghi vấn. Xem evidence trước khi kết luận nguyên nhân.

In [ ]:
pred = read_jsonl(run / 'predictions/ml_logistic-s42.jsonl')
print(pred[0])
summary = read_json(run / 'summary.json')
for algorithm, result in summary.items():
    print(algorithm, result['aggregate']['mrr'], result['gate']['decision'])


## 5 Thử experiment mới
Dùng terminal chạy `python -m rca_bench demo --config configs/demo.json --output runs/new_demo`. Muốn thay modality, chạy `ablate`. Khi thay dữ liệu thật, đọc `docs/03_data_contract.md` và `docs/05_runbook.md` trước.